In [1]:
import pandas as pd

df = pd.read_csv('desastres_reports.csv')

print(df.head())


   report_id            timestamp   latitude  longitude      source_type  \
0          1  2025-05-17 13:50:48 -23.466465 -46.835409      App Usuário   
1          2  2025-05-11 05:03:27 -23.916820 -46.718736  Equipe de Campo   
2          3  2025-05-14 19:29:17 -23.144356 -46.700391      App Usuário   
3          4  2025-05-31 05:21:42 -23.439524 -46.557279      App Usuário   
4          5  2025-06-01 18:07:22 -23.512968 -46.177304      App Usuário   

  observation_type severity_reported  num_affected_estimate  \
0       Alagamento           Crítica                    186   
1     Deslizamento             Baixa                     11   
2     Deslizamento             Média                    110   
3         Incêndio              Alta                    163   
4     Deslizamento             Baixa                     46   

  infrastructure_damage accessibility  \
0                  Leve       Difícil   
1                Severo       Difícil   
2                Nenhum         Fácil   


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Seleciona colunas relevantes
features = ['source_type', 'observation_type', 'severity_reported',
            'num_affected_estimate', 'infrastructure_damage', 'accessibility']
target = 'AREA_RISK_CLASSIFICATION'

# Codifica colunas categóricas
df_encoded = df.copy()
label_encoders = {}

for col in features + [target]:
    if df_encoded[col].dtype == 'object':
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le

# Split dos dados
X = df_encoded[features]
y = df_encoded[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Treina modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Aplica predição no DataFrame original
df['predicted_class'] = label_encoders[target].inverse_transform(model.predict(X))


In [4]:
import geopandas as gpd
from shapely.geometry import Point

# Cria pontos geográficos
geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
geo_df = gpd.GeoDataFrame(df, geometry=geometry)

# Define sistema de coordenadas
geo_df.set_crs(epsg=4326, inplace=True)


,report_id,timestamp,latitude,longitude,source_type,observation_type,severity_reported,num_affected_estimate,infrastructure_damage,accessibility,additional_notes,AREA_RISK_CLASSIFICATION,predicted_class,geometry
0,1,2025-05-17 13:50:48,-23.466465,-46.835409,App Usuário,Alagamento,Crítica,186,Leve,Difícil,Adipisci doloribus totam id.,Monitorar,Monitorar,POINT (-46.83541 -23.46646)
1,2,2025-05-11 05:03:27,-23.916820,-46.718736,Equipe de Campo,Deslizamento,Baixa,11,Severo,Difícil,Aliquid ab libero adipisci molestias ipsam dig...,Risco Imediato,Risco Imediato,POINT (-46.71874 -23.91682)
2,3,2025-05-14 19:29:17,-23.144356,-46.700391,App Usuário,Deslizamento,Média,110,Nenhum,Fácil,Adipisci quis quibusdam minima molestiae non q...,Suporte Necessário,Suporte Necessário,POINT (-46.70039 -23.14436)
3,4,2025-05-31 05:21:42,-23.439524,-46.557279,App Usuário,Incêndio,Alta,163,Severo,Difícil,Iste adipisci repellat.,Risco Imediato,Risco Imediato,POINT (-46.55728 -23.43952)
4,5,2025-06-01 18:07:22,-23.512968,-46.177304,App Usuário,Deslizamento,Baixa,46,Moderado,Bloqueado,Inventore harum rem magnam expedita accusantiu...,Atenção Urgente,Atenção Urgente,POINT (-46.1773 -23.51297)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,2025-05-21 05:39:15,-23.107299,-46.922707,Equipe de Campo,Estrutura Danificada,Alta,72,Leve,Difícil,Velit ullam fuga quo dicta delectus reprehende...,Suporte Necessário,Atenção Urgente,POINT (-46.92271 -23.1073)
996,997,2025-05-04 00:52:28,-23.243410,-46.532586,Equipe de Campo,Incêndio,Crítica,46,Severo,Fácil,Occaecati vitae animi alias molestiae magni si...,Monitorar,Monitorar,POINT (-46.53259 -23.24341)
997,998,2025-05-27 20:07:01,-23.713774,-46.109052,App Usuário,Incêndio,Alta,176,Moderado,Bloqueado,Praesentium iste reiciendis architecto molliti...,Monitorar,Monitorar,POINT (-46.10905 -23.71377)
998,999,2025-05-05 10:54:05,-23.795903,-46.436183,Drone,Deslizamento,Baixa,177,Severo,Fácil,Alias ipsa eius veritatis esse.,Monitorar,Risco Imediato,POINT (-46.43618 -23.7959)


In [5]:
import plotly.express as px

# Mapbox requer uma chave (para mapa bonito). Você pode usar sem, mas com limitações.
px.set_mapbox_access_token("SEU_TOKEN_MAPBOX_AQUI")  # Ou comente essa linha

fig = px.scatter_mapbox(
    geo_df,
    lat=geo_df.geometry.y,
    lon=geo_df.geometry.x,
    color='predicted_class',
    hover_data=['observation_type', 'severity_reported', 'num_affected_estimate', 'additional_notes'],
    title="Classificação de Áreas de Risco - Modelo Preditivo",
    zoom=9,
    height=700
)

fig.update_layout(mapbox_style="open-street-map")  # sem token
fig.show()
